In [4]:
import torch 
import pandas as pd 
import torch.nn as nn 
import torch.optim as optim 
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# !pip install transformers

In [5]:
# 데이터를 로드 ratings_train.txt 로드 
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.dropna(inplace=True)
df.drop_duplicates('document', inplace=True)
df = df[:1000]

In [6]:
# 토크나이저 로드 
tokenizer = AutoTokenizer.from_pretrained('beomi/kcbert-base')

c:\Users\ekfla\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ekfla\.cache\huggingface\hub\models--beomi--kcbert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [8]:
X = df['document'].tolist()
y = df['label'].tolist()
# AutoTokenizer를 이용해서 불러온 토큰화 함수는 입력값을 리스트형태로 받는다. 
tokenized_inputs = tokenizer(
    X, 
    padding = 'max_length', 
    max_length = 64, 
    truncation = True, 
    return_tensors = 'pt'
)

In [13]:
tokenized_inputs

{'input_ids': tensor([[    2,  2170,   832,  ...,     0,     0,     0],
        [    2,  3521,    17,  ...,     0,     0,     0],
        [    2,  8069,  4089,  ...,     0,     0,     0],
        ...,
        [    2,  2177,  4970,  ...,     0,     0,     0],
        [    2,  2635,  4455,  ...,     0,     0,     0],
        [    2,  8451, 24750,  ...,  4327,    17,     3]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]])}

In [14]:
class SampleDataset(Dataset):
    def __init__(self, tokenized_datas, labels):
        # tokenized_datas : AutoTokenizer를 이용하여 토큰화한 데이터 (dict)
        # labels : 종속 변수 데이터 
        self.input_ids = tokenized_datas['input_ids']
        self.attention_mask = tokenized_datas['attention_mask']
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.attention_mask[idx], self.labels[idx]

In [15]:
# DataLoader 생성 
train_dataset = SampleDataset( tokenized_inputs, y )

train_loader = DataLoader(train_dataset, batch_size = 16, shuffle=True)

In [ ]:
class TransformerCLF(nn.Module):
    def __init__(
            self, vocab_size, d_model=128, nhead = 4, num_layer = 2, num_classes = 2 
    ):
        # vocab_size -> 단어 사전의 길이
        # d_model -> 태랜스포머 인코딩에 입력 차원의 수 
        # nhead -> 헤드의 개수 (attetion 갑을 구하기 위한 시점의 개수), 
        # num_layer -> 레이어의 개수 
        # 임베딩 : 인코딩된 데이터를 벡터화 작업
        self.emb = nn.Embedding(vocab_size, d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers = num_layer)

        self.fc = nn.Linear(d_model, num_classes)